# 📓 Notebook 03 — ส่งข้อมูล SMC ให้ GPT-4o วิเคราะห์

## ในบทนี้คุณจะได้เรียนรู้:
- สร้าง `ChatOpenAI` เพื่อคุยกับ GPT-4o
- ส่งข้อมูล SMC เป็น text ให้ LLM วิเคราะห์
- ใช้ **GPT-4o Vision** ส่งทั้ง text + รูปกราฟ

> ⚠️ ต้องมี `OPENAI_API_KEY` ใน `.env`

In [ ]:
# Setup
%pip install langchain langchain-openai python-dotenv smartmoneyconcepts matplotlib -q

import os, io, base64
import pandas as pd, numpy as np
import smartmoneyconcepts as smc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from IPython.display import Image, display

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
print(f'✅ Model: {OPENAI_MODEL}')

In [ ]:
# ========== Helper Functions (จาก NB01) ==========

def get_mt5_data(symbol='XAUUSD', timeframe_str='15m', count=500):
    try:
        import MetaTrader5 as mt5
        if not mt5.initialize(): return None
        tf_map = {'1m': mt5.TIMEFRAME_M1, '5m': mt5.TIMEFRAME_M5, '15m': mt5.TIMEFRAME_M15,
                  '1h': mt5.TIMEFRAME_H1, '4h': mt5.TIMEFRAME_H4, '1d': mt5.TIMEFRAME_D1, '1w': mt5.TIMEFRAME_W1}
        rates = mt5.copy_rates_from_pos(symbol, tf_map.get(timeframe_str, mt5.TIMEFRAME_M15), 0, count)
        mt5.shutdown()
        if rates is None: return None
        df = pd.DataFrame(rates)
        df['date'] = pd.to_datetime(df['time'], unit='s')
        df = df.rename(columns={'tick_volume':'volume'})
        return df[['date','open','high','low','close','volume']].copy()
    except: return None

def get_data(timeframe='15m', count=200):
    df = get_mt5_data(timeframe_str=timeframe, count=count)
    if df is not None: return df
    if os.path.exists('sample_xauusd.csv'):
        return pd.read_csv('sample_xauusd.csv', parse_dates=['date']).tail(count).reset_index(drop=True)
    np.random.seed(42); base=2650; prices=base+np.cumsum(np.random.randn(count)*2.5)
    data=[{'date':pd.Timestamp('2025-01-01')+pd.Timedelta(minutes=15*i),'open':round(p+np.random.randn()*1.5,2),
           'high':round(max(p+np.random.randn()*1.5,p)+abs(np.random.randn())*3,2),
           'low':round(min(p+np.random.randn()*1.5,p)-abs(np.random.randn())*3,2),
           'close':round(p,2),'volume':int(abs(np.random.randn())*1000+500)} for i,p in enumerate(prices)]
    return pd.DataFrame(data)

def compute_indicators(df):
    sl = 20 if len(df)>=200 else 15 if len(df)>=100 else 10 if len(df)>=50 else max(5,len(df)//6)
    swing = smc.smc.swing_highs_lows(df, swing_length=sl)
    return {'swing':swing, 'fvg':smc.smc.fvg(df,join_consecutive=False),
            'bos_choch':smc.smc.bos_choch(df,swing,close_break=True),
            'ob':smc.smc.ob(df,swing,close_mitigation=False),
            'liquidity':smc.smc.liquidity(df,swing,range_percent=0.02)}

def summarize_smc(df, ind, tf='15m'):
    last=df.iloc[-1]; fvg=ind['fvg']; bc=ind['bos_choch']; ob_d=ind['ob']; liq=ind['liquidity']
    lines=[f'=== SMC Analysis ({tf}, {len(df)} candles) ===',f'Price: {last["close"]:.2f}']
    lines.append(f'FVG: Bull={((fvg["FVG"]==1)&fvg["MitigatedIndex"].isna()).sum()}, Bear={((fvg["FVG"]==-1)&fvg["MitigatedIndex"].isna()).sum()}')
    bm=bc['BOS'].notna()
    if bm.any():
        lb=bc[bm].iloc[-1]; lines.append(f'Last BOS: {"Bull" if lb["BOS"]==1 else "Bear"} @ {lb["Level"]:.2f}')
    cm=bc['CHOCH'].notna()
    if cm.any():
        lc=bc[cm].iloc[-1]; lines.append(f'Last CHOCH: {"Bull" if lc["CHOCH"]==1 else "Bear"} @ {lc["Level"]:.2f}')
    lines.append(f'OB: Bull={((ob_d["OB"]==1)&ob_d["MitigatedIndex"].isna()).sum()}, Bear={((ob_d["OB"]==-1)&ob_d["MitigatedIndex"].isna()).sum()}')
    lines.append(f'Liquidity: BSL={((liq["Liquidity"]==1)&liq["Swept"].isna()).sum()}, SSL={((liq["Liquidity"]==-1)&liq["Swept"].isna()).sum()}')
    return '\n'.join(lines)

print('✅ Helper functions พร้อมแล้ว')

## 🤖 Cell 3: สร้าง LLM และส่ง SMC Data ให้วิเคราะห์

In [ ]:
# สร้าง LLM
llm = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=0.1, max_tokens=2000)

# ดึงข้อมูล + คำนวณ indicators
df = get_data('15m', 200)
ind = compute_indicators(df)
smc_text = summarize_smc(df, ind, '15m')
print('📊 ข้อมูล SMC:')
print(smc_text)

# ส่งให้ LLM
print('\n🤖 กำลังส่งให้ GPT วิเคราะห์...')
messages = [
    SystemMessage(content='คุณคือ SMC Trading Analyst เชี่ยวชาญ ICT methodology ตอบภาษาไทย'),
    HumanMessage(content=f'ข้อมูล SMC ของ XAUUSD:\n\n{smc_text}\n\nแนะนำ BUY/SELL/HOLD พร้อม Entry, SL, TP'),
]
response = llm.invoke(messages)
print('\n📥 ผลการวิเคราะห์:')
print(response.content)

## 🖼️ Cell 4: GPT-4o Vision — ส่งกราฟ + text

ให้ AI "เห็น" กราฟจริงๆ ผลลัพธ์จะแม่นยำกว่า text อย่างเดียว

In [ ]:
plt.style.use('dark_background')

def create_chart_base64(df, ind, last_n=50):
    """สร้างกราฟ → return base64 string สำหรับส่ง Vision"""
    start = max(0, len(df)-last_n)
    w = df.iloc[start:]; n = len(w)
    fvg_w = ind['fvg'].iloc[start:]; ob_w = ind['ob'].iloc[start:]
    bc_w = ind['bos_choch'].iloc[start:]; sw_w = ind['swing'].iloc[start:]
    
    fig, ax = plt.subplots(figsize=(14,6), facecolor='#0C0E12')
    ax.set_facecolor('#0C0E12')
    o,h,l,c = w['open'].values, w['high'].values, w['low'].values, w['close'].values
    for i in range(n):
        col = '#77dd76' if c[i]>=o[i] else '#ff6962'
        ax.plot([i,i],[l[i],h[i]],color=col,lw=1)
        ax.add_patch(Rectangle((i-.3,min(o[i],c[i])),.6,max(abs(c[i]-o[i]),.1),facecolor=col,alpha=.8))
    
    for i in range(len(fvg_w)):
        if pd.notna(fvg_w['FVG'].iloc[i]) and fvg_w['FVG'].iloc[i]!=0 and pd.isna(fvg_w['MitigatedIndex'].iloc[i]):
            ax.add_patch(Rectangle((i,fvg_w['Bottom'].iloc[i]),n-i,fvg_w['Top'].iloc[i]-fvg_w['Bottom'].iloc[i],facecolor='yellow',alpha=.2,lw=.5))
    for i in range(len(ob_w)):
        if pd.notna(ob_w['OB'].iloc[i]) and ob_w['OB'].iloc[i]!=0 and pd.isna(ob_w['MitigatedIndex'].iloc[i]):
            cc='purple' if ob_w['OB'].iloc[i]==1 else 'magenta'
            ax.add_patch(Rectangle((i,ob_w['Bottom'].iloc[i]),n-i,ob_w['Top'].iloc[i]-ob_w['Bottom'].iloc[i],facecolor=cc,alpha=.25,lw=1))
    
    ax.set_xlim(-.5,n-.5); ax.set_ylim(l.min()*.999,h.max()*1.001)
    ax.set_title('XAUUSD SMC Chart',color='white',fontsize=12)
    ax.tick_params(colors='white',labelsize=7); ax.grid(False)
    plt.tight_layout()
    
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight', facecolor='#0C0E12')
    buf.seek(0); plt.close()
    return f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"


# สร้างกราฟ
chart_b64 = create_chart_base64(df, ind, last_n=50)

# แสดงกราฟใน notebook
display(Image(data=base64.b64decode(chart_b64.split(',')[1]), width=800))

# ส่งให้ Vision
print('\n🤖 กำลังส่งให้ GPT-4o Vision...')
vision_llm = ChatOpenAI(model='gpt-4o', api_key=OPENAI_API_KEY, temperature=0.1, max_tokens=2000)
vision_msg = HumanMessage(content=[
    {'type':'text','text':f'SMC Analyst วิเคราะห์ XAUUSD:\n{smc_text}\n\n'
     'ดูกราฟ + ข้อมูล → BUY/SELL/HOLD พร้อม Entry, SL, TP'},
    {'type':'image_url','image_url':{'url':chart_b64,'detail':'high'}}
])
result = vision_llm.invoke([vision_msg])
print('\n📥 Vision Analysis:')
print(result.content)

## ✅ สรุป Notebook 03

1. ✅ `ChatOpenAI` + text analysis
2. ✅ GPT-4o Vision: กราฟ + text → การวิเคราะห์ที่แม่นยำ
3. ✅ ฟังก์ชัน `create_chart_base64()` สำหรับ Vision

---
**➡️ ต่อไป**: `04_agent_with_tools.ipynb` — สร้าง Agent ที่ดึงข้อมูลเองได้